## Extraction
We extract data from the Mario Wiki to create the JSON template. 
This manifests in `extract_level_links`, which works based on the game, and extracts from the table on the wiki page. Since all games have a similar structure, it allows for the data to be extracted quickly, with then only a few manual ammendments.

We do this on NSMBU (Original and Luigi Version), 3D World, and Toad's Treasure Tracker to get the named levels. 

In [1]:
import requests
import re
from bs4 import BeautifulSoup
import json

In [68]:
JSON_TEMPLATE = {
  "meta": {
    "game": "",
    "level_code": ""
  },
  "input": {
    "world": "",
    "layout_description": "",
    "setting": [],
    "enemies": [],
    "obstacles": [],
    "boss": ""
  },
  "output": {
    "name": ""
  }
}

def extract_enemies(soup):
    enemies_header = soup.find("span", id=re.compile("enemies", re.IGNORECASE))
    if not enemies_header:
        return []

    header_h2 = enemies_header.find_parent("h2")
    if not header_h2:
        return []

    enemies = []
    sibling = header_h2.find_next_sibling()

    while sibling:
        if sibling.name == "ul":
            enemies.extend(extract_enemies_from_gallery(sibling))
            break

        if sibling.name == "table":
            enemies.extend(extract_enemies_from_table(sibling))
            break

        sibling = sibling.find_next_sibling()

    return enemies

def extract_enemies_from_gallery(gallery_ul):
    enemies = []
    for item in gallery_ul.find_all("li", class_="gallerybox"):
        text_div = item.find("div", class_="gallerytext")
        if not text_div:
            continue

        name_tag = text_div.find("a")
        if not name_tag:
            continue

        name = name_tag.get_text(strip=True)
        name = clean_enemy_name(name)
        enemies.append(name)

    return enemies

def extract_enemies_from_table(table):
    enemies = []

    for row in table.find_all("tr")[1:]:  
        cols = row.find_all("td")
        if len(cols) < 2:
            continue

        name_tag = cols[1].find("a")
        if name_tag:
            name = name_tag.get_text(strip=True)
            name = clean_enemy_name(name)
            enemies.append(name)

    return enemies

def clean_enemy_name(name):
    return re.sub(r"\s*\(\d+\)$", "", name).strip()

def get_soup(url):
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")

def extract_table(table, game):
    level_links = []
    full_url = "<temp>"

    for row in table.find_all("tr")[1:]:
        cols = row.find_all("td")
        if len(cols) < 2:
            continue

        try:
            level_td = cols[1]

            if game == "New Super Mario Bros. Wii":
                if len(cols) > 3:
                    desc = cols[3].get_text(" ", strip=True)
                else:
                    desc = cols[2].get_text(" ", strip=True)
            else:
                desc = cols[2].get_text(" ", strip=True)

            desc = desc.replace("Read more...", "")

            link_tag = level_td.find("a")
            if not link_tag or "href" not in link_tag.attrs:
                continue

            link = link_tag["href"]
            title = link_tag.get_text(strip=True)
            full_url = "https://mario.wiki" + link
            print(full_url)

            # fetch page
            level_soup = get_soup(full_url)
            if not level_soup:
                continue

            # parse infobox
            info = level_soup.find("table", class_="infobox").find_all("tr")
            if game == "New Super Mario Bros. Wii" or "New Super Mario Bros. 2":
                level_code = info[1].find_all("td")[0].get_text(strip=True)
            else:
                level_code = info[3].find_all("td")[1].get_text(strip=True)
            world = info[4].find_all("td")[1].get_text(strip=True)

            # extract enemies
            enemies = extract_enemies(level_soup)

            level_links.append({
                "meta": {
                    "game": game,
                    "level_code": level_code,
                    "link": full_url
                },
                "input": {
                    "world": world,
                    "layout_description": desc,
                    "setting": [],
                    "enemies": enemies,
                    "obstacles": [],
                    "boss": ""
                },
                "output": {
                    "name": title
                }
            })

        except Exception as e:
            print(f"⚠️ Skipping due to error on {full_url}: {e}")
            continue

    return level_links

def extract_level_links(url, game="NSMBU"):
    soup = get_soup(url)

    if game == "NSMBU":
        tables = soup.find_all("table", class_="wikitable")
        level_jsons = []
        level_jsons += extract_table(tables[0], "New Super Mario Bros. U")
        level_jsons += extract_table(tables[1], "New Super Luigi U")

        return level_jsons
    elif game == "3DW":
        table = soup.find("table", class_="wikitable")
        level_jsons = []
        level_jsons += extract_table(table, "Super Mario 3D World")

        return level_jsons
    elif game == "CTTT":
        table = soup.find("table", class_="wikitable")
        level_jsons = []
        level_jsons += extract_table(table, "Captain Toad's Treasure Tracker")
        
        return level_jsons

    elif game == "NSMB":
        table = soup.find("table", class_="wikitable")
        level_jsons = []
        level_jsons += extract_table(table, "New Super Mario Bros.")

        return level_jsons
    
    elif game == "NSMBWii":
        table = soup.find("table", class_="wikitable")
        level_jsons = []
        level_jsons += extract_table(table, "New Super Mario Bros. Wii")

        return level_jsons
    
    elif game == "NSMB2":
        table = soup.find("table", class_="wikitable")
        level_jsons = []
        level_jsons += extract_table(table, "New Super Mario Bros. 2")

        return level_jsons


In [ ]:
links = []

page_urls = [
    "https://www.mariowiki.com/Acorn_Plains",
    "https://www.mariowiki.com/Layer-Cake_Desert",
    "https://www.mariowiki.com/Sparkling_Waters",
    "https://www.mariowiki.com/Frosted_Glacier",
    "https://www.mariowiki.com/Soda_Jungle",
    "https://www.mariowiki.com/Rock-Candy_Mines",
    "https://www.mariowiki.com/Meringue_Clouds",
    "https://www.mariowiki.com/Peach%27s_Castle_(world)",
    "https://www.mariowiki.com/Superstar_Road",
    
]

for page_url in page_urls:
    links += extract_level_links(page_url, game="NSMBU")

page_urls = [
    "https://www.mariowiki.com/World_1_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_2_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_3_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_4_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_5_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_6_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_Castle",
    "https://www.mariowiki.com/World_Bowser",
    "https://www.mariowiki.com/World_Star_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_Mushroom_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_Flower_(Super_Mario_3D_World)",
    "https://www.mariowiki.com/World_Crown"
]

for page_url in page_urls:
    links += extract_level_links(page_url, game="3DW")

page_urls = [
    "https://www.mariowiki.com/Episode_1",
    "https://www.mariowiki.com/Episode_2",
    "https://www.mariowiki.com/Episode_3"
]

for page_url in page_urls:
    links += extract_level_links(page_url, game="CTTT")

with open("named_levels(temp).json", "w") as f:
    json.dump(links, f)

In [73]:
with open("named_levels.json", "r", encoding="utf-8") as f:
    data = json.load(f)

settings = set()
for level in data:
    for setting in level['input']['obstacles']:
        settings.add(setting)

setting_list = list(settings)
setting_list.sort()
for s in setting_list:
    print(s)

Arrow Lift
Beep Blocks
Big Blocks
Big Pipes
Blackout
Blast Blocks
Bounce Mushrooms
Bouncy Clouds
Brick Blocks
Brick Pillasrs
Bridges
Burners
Cannon
Cannon Boxes
Cat Power Up
Chains
Clear Pipe Cannon
Clear Pipes
Climbing Walls
Cloud Platforms
Cogs
Collapsing Bridge
Conveyor Belts
Crate
Crates
Cross Lifts
Crystal Pipes
Crystals
Dark
Dash Panels
Disappearing Platforms
Donut Blocks
Double Cherry
Elastic Platforms
Fence Grating
Ferris Wheel Platforms
Fire Bars
Flip Panels
Floating Platforms
Flying Carpet
Geyser Pipe
Giant Blocks
Giant Cogs
Giant Skewer
Girders
Hammer Platforms
Honeycomb Platforms
Huge Icicle
Ice Blocks
Ice Skate
Icicle
Invisible Path
Iron Blocks
Ladder
Ladders
Lava
Lava Waves
Leaf Platforms
Lean Platform
Lift
Light Blocks
Limited Lift
Limited Raft
Mecha Hand
Mega Mushroom
Meteors
Minecart
Movable Blocks
Moving Platforms
Moving Water
Multi-person Platform
Mushroom Platforms
Mushrooms
Noteblocks
P-Switches
POW Blocks
Plessie
Poison
Poles
Posion
Propeller Platform
Quicksand
Ra

In [69]:
links = []

# page_urls = [
#     "https://www.mariowiki.com/World_1_(New_Super_Mario_Bros.)",
#     "https://www.mariowiki.com/World_2_(New_Super_Mario_Bros.)",
#     "https://www.mariowiki.com/World_3_(New_Super_Mario_Bros.)",
#     "https://www.mariowiki.com/World_4_(New_Super_Mario_Bros.)",
#     "https://www.mariowiki.com/World_5_(New_Super_Mario_Bros.)",
#     "https://www.mariowiki.com/World_6_(New_Super_Mario_Bros.)",
#     "https://www.mariowiki.com/World_7_(New_Super_Mario_Bros.)",
#     "https://www.mariowiki.com/World_8_(New_Super_Mario_Bros.)"
# ]

# for page_url in page_urls:
#     links += extract_level_links(page_url, game="NSMB")

page_urls = [
    "https://www.mariowiki.com/World_1_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_2_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_3_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_4_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_5_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_6_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_7_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_8_(New_Super_Mario_Bros._Wii)",
    "https://www.mariowiki.com/World_9_(New_Super_Mario_Bros._Wii)",
]

for page_url in page_urls:
    print(page_url)
    links += extract_level_links(page_url, game="NSMBWii")

page_urls = [
    "https://www.mariowiki.com/World_1_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_2_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_Mushroom_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_3_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_4_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_Flower_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_5_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_6_(New_Super_Mario_Bros._2)",
    "https://www.mariowiki.com/World_Star_(New_Super_Mario_Bros._2)",
]

for page_url in page_urls:
    links += extract_level_links(page_url, game="NSMB2")

https://www.mariowiki.com/World_1_(New_Super_Mario_Bros._Wii)
https://mario.wiki/Peach%27s_Castle#New_Super_Mario_Bros._Wii
https://mario.wiki/World_1-1_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_1-2_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_1-3_(New_Super_Mario_Bros._Wii)
https://mario.wiki/Warp_Cannon
https://mario.wiki/World_1-Tower_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_1-4_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_1-5_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_1-6_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_1-Castle_(New_Super_Mario_Bros._Wii)
https://www.mariowiki.com/World_2_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_2-1_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_2-2_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_2-3_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_2-Tower_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_2-4_(New_Super_Mario_Bros._Wii)
https://mario.wiki/World_2-5_

In [67]:
soup = get_soup("https://www.mariowiki.com/World_2_(New_Super_Mario_Bros._Wii)")
table = soup.find("table", class_="wikitable")
level_jsons = []
level_jsons += extract_table(table, "New Super Mario Bros. Wii")

3
https://mario.wiki/World_2-1_(New_Super_Mario_Bros._Wii)
3
https://mario.wiki/World_2-2_(New_Super_Mario_Bros._Wii)


KeyboardInterrupt: 

In [72]:
with open("unnamed_levels(temp).json", "w") as f:
    json.dump(links, f, indent=4)

In [70]:
from pprint import pprint
pprint(links)

[{'input': {'boss': '',
            'enemies': [],
            'layout_description': "Peach's Castle is the first location in the "
                                  'map. The player can buy and watch hint '
                                  'movies here.',
            'obstacles': [],
            'setting': [],
            'world': 'Mushroom Kingdom(Toad Town)'},
  'meta': {'game': 'New Super Mario Bros. Wii',
           'level_code': "Peach's Castle inSuper Mario Odyssey",
           'link': 'https://mario.wiki/Peach%27s_Castle#New_Super_Mario_Bros._Wii'},
  'output': {'name': 'World 1-'}},
 {'input': {'boss': '',
            'enemies': ['Goomba', 'Koopa Troopa'],
            'layout_description': 'The first level of New Super Mario Bros. '
                                  'Wii . It introduces the Propeller Suit , '
                                  'Goombas and Koopa Troopas . ',
            'obstacles': [],
            'setting': [],
            'world': 'New Super Mario Bros. Wii